# Noise Patterns (Blacklist)

In [0]:
import re
from pyspark.sql.functions import udf, col, when, size, concat_ws
from pyspark.sql.types import ArrayType, StringType

# ── Blacklist patterns (case-insensitive substring match) ─────
NOISE_PATTERNS = [
    # Location / address
    "located", "location:", "opposite ", "next to", "near",
    "situated", "found in", "address", "based in", "serves"
    "headquarters", "headquarter", "head office",
    "country", "city", "region", "district", "is a",

    # Business/LinkedIn profile scraping
    "company size", "company type", "industry",
    "type", "common name"
    "employees", "staff size",
    "founded and managed", "founded by", "managed by",
    "under the guidance of", "under guidance",
    "ownership", "owned by", "has a Memorandum of Understanding",
    "has a MoU"

    # Contact info
    "telephone", "phone:", "tel:", "tel.", "mobile:",
    "contact number", "contact us", "call us",
    "email:", "e-mail:", "mail us",
    "organization name", "organization abbreviation",
    "whatsapp",

    # Web / social / directory
    "website", "web", "visit us", "visit our",
    "http", "www.", ".com", ".org", ".gh", ".net",
    "listed as", "listed in categories", "related place",
    "directory", "ghanabusinessweb", "businessweb",
    "last updated", "page", "see more",
    "facebook", "twitter", "instagram", "linkedin",
    "follow", "like", "likes", "followers",

    # Admin / registration
    "registered", "registration", "identified"
    "established", "created on", "part of", "affiliated"
    "affiliated", "affiliation", "promotes"
    "mission", "vision", "values", "member"

    # Business metrics / voting
    "voted", "award", "ranked",
    "reviews", "review", "rating",
    "partnerships", "partnership",  # but keep "accepts insurance"
    "app available","appointment",

    # Pricing / hours (operational, not medical)
    "open from", "opening hours", "hours", "closed on",
    "fee", "fees", "charges", "prices", "price"
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday",

    # Size/employee counts
    "employees", "employee"

    # Ghana-Specific Addresses & Post
    "p.o. box", "po box", "post office box", "private mail bag", "pmb",
    "digital address", "gps", "longitude", "latitude",
    "directions", "map",

    # Generic Marketing Fluff & Intros
    "welcome", "state of the art", "world class", "world-class",
    "leading provider", "goal", "mission", "we are committed to",
    "dedicated team", "strives to", "best in class", "proudly serving",

    # Non-Clinical Amenities
    "parking available", "free parking", "car park", "valet",
    "wifi", "wi-fi", "canteen", "cafeteria", "waiting area",

    # Payments & Finance (Non-Insurance)
    "mobile money", "visa", "mastercard", "cash only",
    "payment methods", "we accept",

    # Web / UI Artifacts & Legal
    "click here", "read more", "load more", "learn more", 
    "copyright", "all rights reserved", "terms of use", "privacy policy",
    "subscribe", "newsletter"

    # ── Sneaky Directory & Web Scraping ──
    "people like this", "people follow this", "like this page", "follow this page",
    "ghanayello", "listed in the", "category for", "registered on",
    
    # ── Vague Marketing & Scale ──
    "happy patients", "thousands of patients", "patients annually",
    "comprehensive healthcare destination", "positions itself", 
    "attracts international", "expanding from", "collaborates with",
    
    # ── Admin, History & NGO Fluff ──
    "founded in", "started in", "mission to", "vision to", 
    "missionary", "ngo", "non-profit", "non profit", "church-based",
    
    # ── Database Junk & Hours ──
    "currently closed", "system codes", "entityid", "orgmastid"
]

# Medical keywords Patterns (Whitelist)

In [0]:
MEDICAL_WHITELIST = [
    # Clinical services
    "dialysis", "hemodialysis", "haemodialysis",
    "surgery", "surgical", "operation", "theatre", "theater",
    "icu", "intensive care", "critical care", "nicu",
    "emergency", "casualty", "trauma", "resuscitat",
    "maternity", "obstetric", "labour", "labor", "delivery",
    "antenatal", "postnatal", "antenatal", "medicine",
    "laboratory", "lab test", "blood test", "pathology",
    "pharmacy", "dispensary", "medication",
    "radiology", "imaging", "mri", "ct scan", "x-ray",
    "xray", "ultrasound", "mammograph", "scan",
    "ophthalmology", "eye care", "optometry", "cataract",
    "dental", "dentist", "oral health", "internal medicine", "internalMedicine"
    "physiotherapy", "rehabilitation", "therapy",
    "vaccination", "immunization", "vaccine",
    "blood bank", "blood transfusion",
    "mortuary", "post mortem", "disease", "illness", "illnesses",
    "infection", "infections", "diseases"
    "oncology", "chemotherapy", "radiotherapy", "cancer",
    "cardiology", "cardiac", "ecg", "echocardiograph",
    "neurology", "neurological",
    "pediatric", "paediatric", "child health",
    "hiv", "aids", "pmtct", "art ",
    "mental health", "psychiatry", "counseling",
    "nhis accredited","nhis-accredited", "nhis accreditation",  # real administrative signal
    "government-owned", "government owned",   # ownership type is useful
    "public hospital", "district hospital", "regional hospital",
    "teaching hospital", "specialist hospital",
    "health centre", "health center",
    "24/7", "24-hour", "24 hour", "round the clock",  # only specific 24h service claims
    "general services", "general medicine", "outpatient", "inpatient",
    "anesthesia", "anaesthesia",
    "wound care", "dressing",
    "family planning",
    "nutrition", "dietitian",
    "stroke", "seizure",
    "fracture", "orthopedic", "orthopaedic",
    "ambulance",
    
    # Endemic & Infectious Diseases (Crucial for Ghana)
    "malaria", "tuberculosis", "tb clinic", "dots", 
    "typhoid", "cholera", "yellow fever", "infectious disease",
    "hepatitis", "sti", "std", "leprosy", "buruli ulcer",

    # Key Medical Subspecialties missing
    "gynecology", "gynaecology", "ent", "ear nose throat", "ear, nose",
    "dermatology", "skin clinic", "urology", "nephrology", 
    "endocrinology", "diabetes", "diabetic clinic", "gastroenterology", 
    "pulmonology", "respiratory", "rheumatology",

    # Expanded Diagnostics & Procedures
    "endoscopy", "colonoscopy", "biopsy", "fluoroscopy", 
    "eeg", "dialysis", # (Note: you have dialysis, just ensuring it's comprehensive)
    
    # Maternal, Neonatal & Women's Health
    "neonatal", "incubator", "reproductive health", "fertility", "ivf",
    "cervical screening", "cervical cancer", "pap smear",
    "child welfare clinic", "cwc",  # CWC is heavily used in Ghana's health system

    # Surgical & Advanced Care
    "plastic surgery", "reconstructive", "neurosurgery", "pediatric surgery",
    "laparoscopy", "laparoscopic", "endoscopic surgery",

    # Alternative / Regulated Medicine (Common in Ghana)
    "homeopathic", "homeopathy", "herbal medicine", "traditional medicine", 
    "plant medicine",

    # Public Health & Allied Health
    "screening", "wellness clinic", "public health", "outreach",
    "prosthetics", "orthotics", "occupational therapy", "speech therapy"
]

# cleaning function

In [0]:

def clean_capability_item(text: str) -> bool:
    """
    Returns True if this capability item should be KEPT.

    Pass 1: Blacklist — remove obvious noise
    Pass 2: Whitelist — keep only items with medical content

    For pre-computation we're aggressive:
    if it doesn't sound medical, it's out.
    """
    if not text:
        return False

    stripped = text.strip()

    # Too short to be meaningful
    if len(stripped) < 6:
        return False

    # Purely numeric (phone numbers, dates, registration numbers)
    if re.match(r'^[\d\s\-\+\(\)\/]+$', stripped):
        return False

    text_lower = stripped.lower()

    # ── Pass 1: Blacklist ─────────────────────────────────────
    if any(pattern in text_lower for pattern in NOISE_PATTERNS):
        return False

    # ── Pass 2: Whitelist ─────────────────────────────────────
    # Item must contain at least one medical keyword to be kept
    if any(kw in text_lower for kw in MEDICAL_WHITELIST):
        return True

    # ── Pass 3: Borderline heuristics ────────────────────────
    # Some items don't match the whitelist but are still medical
    # Keep if: starts with a verb + medical-sounding noun
    # e.g. "Conducts blood pressure checks", "Provides general outpatient care"
    medical_verbs = ["provides", "offers", "conducts", "performs",
                     "treats", "manages", "handles", "specializes",
                     "equipped", "available", "operates"]
    if any(text_lower.startswith(v) for v in medical_verbs):
        # Secondary check: does it mention anything medical-sounding?
        medical_nouns = ["care", "service", "treatment", "patient",
                         "medical", "clinical", "health", "disease",
                         "condition", "ward", "unit", "center", "centre"]
        if any(n in text_lower for n in medical_nouns):
            return True

    # Everything else — discard
    # Being aggressive here is correct:
    # a false negative (discarding real content) is better than
    # a false positive (keeping noise that corrupts anomaly scores)
    return False


def clean_capability_array(items) -> list:
    """
    Input: Python list (from ArrayType column) or NumPy array (from Pandas)
    Output: filtered list with only medical content
    """
    # 1. Handle actual Nulls
    if items is None:
        return []

    # 2. Handle NumPy arrays (caused by .toPandas())
    if hasattr(items, "tolist"):
        items = items.tolist()

    # 3. Handle floats (Pandas sometimes converts Nulls to float NaN)
    if not isinstance(items, list) or len(items) == 0:
        return []

    cleaned = []
    for item in items:
        if not item or not isinstance(item, str):
            continue
        text = item.strip()
        if clean_capability_item(text):
            cleaned.append(text)

    return cleaned

clean_capability_udf = udf(clean_capability_array, ArrayType(StringType()))

# Test the cleaning on your actual data

In [0]:
import pandas as pd

df = spark.read.table("workspace.default.facilities3")

sample = df.select("name", "capability") \
           .filter(col("capability").isNotNull()) \
           .filter(size(col("capability")) > 0) \
           .limit(100)

sample_pd = sample.toPandas()

for _, row in sample_pd.iterrows():
    raw_val = row["capability"]
    
    # 1. Safely convert NumPy arrays, None, and NaNs to standard Python lists
    if raw_val is None or (isinstance(raw_val, float) and pd.isna(raw_val)):
        original = []
    elif hasattr(raw_val, "tolist"):
        original = raw_val.tolist()
    else:
        original = list(raw_val)

    # 2. Apply cleaning
    cleaned  = clean_capability_array(original)
    
    # 3. Safely find removed items (no more 'or []' needed)
    removed  = [x for x in original if x not in cleaned]

    if removed:  # only show rows where something was removed
        print(f"\n{'='*60}")
        print(f"Facility: {row['name']}")
        print(f"\nKEPT ({len(cleaned)}):")
        for item in cleaned:
            print(f"  ✓ {item}")
        print(f"\nREMOVED ({len(removed)}):")
        for item in removed:
            print(f"  ✗ {item}")

# Check specific examples from your question

In [0]:
test_cases = [
    # LinkedIn scraping — should all be removed
    ["Headquarters: Haatso, Ghana",
     "Industry: Hospitals and Health Care",
     "Type: Nonprofit",
     "Company size: 11-50 employees",
     "Specialties: HIV, AIDS, PMTCT, Behavior Change Communication"],

    # Operational noise — mostly removed
    ["Always open",
     "Open 24 hours a day, 7 days a week",
     "Has 11-15 employees",
     "Managed by Dr. Anthony Twum-Barimah"],

    # Real medical content — should be kept
    ["Laboratory tests conducted 24/7",
     "Pharmacy services available 24/7",
     "Dialysis Center on site",
     "ANIH app enables appointment scheduling",
     "Insurance partnerships with Metro, Glico"],

    # Eye hospital — mixed
    ["Voted Best Eye Hospital in Ghana",
     "Founded and managed under the guidance of Dr. Asiwome",
     "Staff includes ophthalmologists, surgeons, and eye care professionals",
     "Has insurance partnerships with leading providers",
     "Eye hospital specializing in ophthalmology"],

    # Health centre — mixed
    ["Abura Health Centre is a Government-owned Health Centre.",
     "Services: General.",
     "NHIS accredited: yes"],
]

print("CLEANING TEST RESULTS:")
for i, test in enumerate(test_cases, 1):
    cleaned = clean_capability_array(test)
    removed = [x for x in test if x not in cleaned]
    print(f"\n--- Test {i} ---")
    print(f"Kept   : {cleaned}")
    print(f"Removed: {removed}")

# Apply and create capability_cleaned column

In [0]:
df = spark.read.table("workspace.default.facilities3")

df = df.withColumn(
    "capability_cleaned",
    clean_capability_udf(col("capability"))
)

df.select(
    "name",
    size(col("capability")).alias("capability_original_count"),
    size(col("capability_cleaned")).alias("capability_cleaned_count")
).withColumn(
    "items_removed",
    col("capability_original_count") - col("capability_cleaned_count")
).orderBy(col("items_removed").desc()).show(20, truncate=False)

df.createOrReplaceTempView("temp_cleaned_facilities")

# Distribution check after cleaning

In [0]:
result_df = spark.sql("""
    SELECT
        SUM(CASE WHEN size(capability)         = 0 THEN 1 ELSE 0 END) AS original_empty,
        SUM(CASE WHEN size(capability_cleaned) = 0 THEN 1 ELSE 0 END) AS cleaned_empty,
        SUM(CASE WHEN size(capability_cleaned) > 0 THEN 1 ELSE 0 END) AS cleaned_has_content,
        COUNT(*) AS total
    FROM temp_cleaned_facilities
""")

display(result_df)

# Write capability_cleaned to the table

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.facilities3")

print("capability_cleaned column added to facilities3")